In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [2]:
corrdone=False

In [3]:
from pyCHX.chx_packages import *
%matplotlib notebook
plt.rcParams.update({'figure.max_open_warning': 0})
plt.rcParams.update({ 'image.origin': 'lower'   })
plt.rcParams.update({ 'image.interpolation': 'none'   })
import pickle as cpk
from pyCHX.chx_xpcs_xsvs_jupyter_V1 import *

from skbeam.core.roi import segmented_rings, ring_edges
from skbeam.core.utils import radial_grid, angle_grid
from skimage.draw import line_aa, line, polygon, ellipse, disk


import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure, plot, imshow
%matplotlib notebook

from pyCHX.chx_packages import *
from pyCHX.chx_xpcs_xsvs_jupyter_V1 import *

import json  ### NEW for auto_masking

%run /nsls2/data/chx/legacy/analysis/2022_3/lwiegart/development/chx_analysis_setup.ipynb


/nsls2/conda/envs/2023-3.2-py310-tiled/lib/python3.10/site-packages/databroker/v1.py:72: UserWarning: In databroker 2.x, there are separate notions of 'server' and 'client', and register_handler(...) has no effect on the client. Likely this is being done for you on the server side, so you should not worry about this message unless you encounter trouble loading large array data.
  warnings.warn(


/nsls2/conda/envs/2023-3.2-py310-tiled srv3
running on: srv3   environment: 2023-3.2-py310-tiled
setting '_base_path_' as /nsls2/data/chx/legacy/analysis/
ran "%run -i /nsls2/data/chx/legacy/analysis/2022_3/lwiegart/development/roi_nr_2019_3_0_1.py" to fix ROI numbering for PHI-sliced data sets. Should get fixed in pyCHX...
ran "%run -i /nsls2/data/chx/legacy/analysis/2023_2/lwiegart/pyCHX_June2023/chx_outlier_detection.py": this should become part of pyCHX.
ran "%run -i /nsls2/data/chx/legacy/analysis/2023_2/lwiegart/pyCHX_June2023/fix_get_sid_filenames.py": this should be fixed and re-deployed in pyCHX.

environment dependent settings and patches:
using "%matplotlib inline" for plotting
ran "%run -i /nsls2/data/chx/legacy/analysis/2022_3/lwiegart/development/polygonmask_fix.py".
ran "%run -i /nsls2/data/chx/legacy/analysis/2023_2/lwiegart/pyCHX_June2023/chx_compress.py": temporary fix for depreciated np.float.
ran "%run -i /nsls2/data/chx/legacy/analysis/2023_2/lwiegart/pyCHX_June202

In [4]:
scat_geometry = 'saxs'

In [5]:
scan_nr=147267

h=db[scan_nr]
uid=h.start['uid']
print('uid: %s'%uid)

uid: fc79b309-78be-4081-a2b6-2167d03060e2


In [6]:
#uid = '445b9d' #(scan num: 135097)
    
uidstr = 'uid=%s'%uid[:6]
print(uidstr)

dscan = False #  True #False
count = not dscan #True #False #True

uid=fc79b3


In [7]:
cycle= '2024_1'  #change clycle here
default_dir = _base_path_
path = default_dir + '%s/masks/'%cycle
username = 'leheny'

data_dir0  = create_user_folder(cycle, username, default_dir=default_dir)
print( data_dir0 )
uid = uid[:8]
print('The current uid for analysis is: %s...'%uid)

mask_path =default_dir+'%s/%s/masks/'%(cycle,username)


Results from this analysis will be stashed in the directory /nsls2/data/chx/legacy/analysis/2024_1/leheny/Results/
/nsls2/data/chx/legacy/analysis/2024_1/leheny/Results/
The current uid for analysis is: fc79b309...


## Load data and ROI

In [8]:
if dscan:## This works for dscan with each motor position having multi frames
    imgs =  np.array(db.get_images(db[uid], 'eiger4m_single_image'))
    #pixel_mask = imgs[0].md['binary_mask']
    #md = imgs[0].md
    md = get_meta_data( uid )
    
    #data = db[uid]
    #detector = get_detector(data)
    #imgs = np.array(db.get_images(fig,ax = plt.subplots(1,3,figsize=(15,5))

    
    
else:
    #imgs =  db.get_images(db[uid], 'eiger1m_single_image')
    md = get_meta_data( uid )
    sud = get_sid_filenames(db[uid])
    for pa in sud[2]:
        if 'master.h5' in pa:
            data_fullpath = pa

    detectors = sorted(get_detectors(db[uid]))
    print('The detectors are:%s'%detectors)
    if len(detectors) >1:
        md['detector'] = detectors[1]
        print( md['detector'])
    if md['detector'] =='eiger4m_single_image' or md['detector'] == 'image':    
        reverse= True
        rot90= False
    elif md['detector'] =='eiger500K_single_image':    
        reverse= True
        rot90=True
    elif md['detector'] =='eiger1m_single_image':    
        reverse= True
        rot90=False
    print('Image reverse: %s\nImage rotate 90: %s'%(reverse, rot90))            
    try:
        cx , cy = md_blue['beam_center_x'], md_blue['beam_center_y']
        print(cx,cy)
    except:
        print('Will find cx,cy later.')  

    imgs = load_data( uid, 'eiger4m_single_image', reverse= reverse, rot90=rot90  )
    #md = imgs.md
    md.update( imgs.md );Nimg = len(imgs);
    pixel_mask = md['binary_mask']    
    
    if scat_geometry =='gi_saxs':
        inc_x0 =  md['beam_center_x'] 
        inc_y0 =  imgs[0].shape[0] - md['beam_center_y']     

        refl_x0 =     md['beam_center_x']  
        refl_y0 =     imgs[0].shape[0] - 673   

        print( "inc_x0, inc_y0, ref_x0,ref_y0 are: %s %s %s %s."%(inc_x0, inc_y0, refl_x0, refl_y0) )
    else:
        if md['detector'] =='eiger4m_single_image' or md['detector'] == 'image' or md['detector']=='eiger1m_single_image':    
            inc_x0 =  imgs[0].shape[0] - md['beam_center_y']   
            inc_y0=   md['beam_center_x']
        elif md['detector'] =='eiger500K_single_image':    
            inc_y0 =  imgs[0].shape[1] - md['beam_center_y']   
            inc_x0 =   imgs[0].shape[0] - md['beam_center_x']

        print(inc_x0, inc_y0)    
    
    
    dpix, lambda_, Ldet,  exposuretime, timeperframe, center = check_lost_metadata(
    md, Nimg, inc_x0 = inc_x0, inc_y0=   inc_y0, pixelsize = 7.5*10*(-5) )
    if scat_geometry =='gi_saxs':center=center[::-1]

    setup_pargs=dict(uid=uidstr, dpix= dpix, Ldet=Ldet, lambda_= lambda_, exposuretime=exposuretime,
            timeperframe=timeperframe, center=center, path= data_dir)
    print_dict( setup_pargs )
    
Nimg = len(imgs);
#if 'number of images'  not in list(md.keys()):
md['number of images']  = Nimg
print( 'The data are: %s' %str(imgs.shape) )
print('metadata: %s'%str(md).split(","))

More than one device. This would have unintented consequences.Currently, only the device contains 'default_dec=eiger'.


JSONDecodeError: Unterminated string starting at: line 1 column 9434 (char 9433)

In [ ]:
str(md).split(",")

In [ ]:
%run  -i pstorage/scan-yh-1.des.py

In [ ]:
# #check information in file header
# DD.FD.FID.seek(0,0)
# buf=DD.FD.FID.read(1024)
# DD.FD.prheader(buf)

In [ ]:
imgs=np.squeeze(imgs)[:,::-1,:]

In [ ]:
figure()
plt.imshow(imgs[0,:,:],vmin=-1,vmax=10)
plt.plot(DD.DD['xbar'],DD.DD['ybar'],'ro')
#beam_center_x 1112.0
#beam_center_y 1224.0
# plot(1112,2167-1224,'bo')
plot(1155,2167-1231,'bo')

In [ ]:
DD.DD

In [ ]:
#if  dscan:
#center = [936, 1147]
center= [2167-DD.DD['xbar'],DD.DD['ybar']]
#md = imgs[0].md
md['detector'] ='eiger4m_single_image'
#dpix, Ldet,lambda_ = 0.075, 16035.64,   1.284411
dpix, Ldet,lambda_ = 0.075, 16035.64,0.96560895
timeperframe = 0.01
exposuretime = timeperframe
setup_pargs=dict(uid=uidstr, dpix= dpix, Ldet=Ldet, lambda_= lambda_, exposuretime=exposuretime,
        timeperframe=timeperframe, center=center, path= data_dir0)

    
        
        

# Partition into bins

In [ ]:
from fitcorrs import *#read in routines
from pyanal.partitions import mkpartlist,partition1d,partition2d
from pyanal.mkphiq import mkphiq
from pyanal.crosscor import *
from pyanal.ldparts import *
from pyanal.h2t import h2t
from pyanal.avgg2 import avgg2
from pyanal.ciravg import *
import scipy.signal
from numpy import r_


In [ ]:
pixellist,qs,phis=mkphiq(DD,phimin=-10)
avgimg=nfc79b309-78be-4081-a2b6-2167d03060e2p.mean(imgs,axis=0)*mask
Q,S=ciravg(avgimg,pixellist,qs,np.arange(45,1000))
SG=np.interp(qs,Q,S)

In [ ]:
qlist=mkpartlist(np.linspace(25,505,20))
philist=mkpartlist(np.arange(0,360,22.5),width=20)
plist,bind,qbind,phibind,noperbin,inpixellist,binlist,binqlist,binphilist=\
    partition2d(pixellist,qs,qlist,phis,philist,thres=100)

In [ ]:
qavg=np.bincount(bind,weights=qs[inpixellist])/noperbin
phiavg=np.bincount(bind,weights=phis[inpixellist])/noperbin

In [ ]:
px=qavg*np.cos(phiavg/57.3)+DD.DD['xbar']
py=qavg*np.sin(phiavg/57.3)+DD.DD['ybar']

In [ ]:
plt.figure()
timg=0*avgimg
timg.ravel()[plist]=phis[inpixellist]
plt.imshow(avgimg,vmax=10)
plt.plot(center[0],center[1],'yo')
plt.plot(px,py,'r.')
plt.plot(DD.DD['xbar'],DD.DD['ybar'],'ro')

In [ ]:
plt.figure()
plt.plot(Q,S)
# for i in range(len(qlist)/2):
#     plot()
sp=np.interp(qlist,Q,S)
spm=(sp[::2]+sp[1::2])/2.0
dq=(qlist[1::2]-qlist[::2])/2
print(dq)
for i in range(len(spm)):
    plt.plot(qlist[[2*i,2*i]],r_[sp[2*i],spm[i]],'r')
    plt.plot(qlist[2*i:2*i+2],r_[spm[i],spm[i]],'r')
    plt.plot(qlist[[2*i+1,2*i+1]],r_[spm[i],sp[2*i+1]],'r')
plt.title(" Q-rings")
plt.ylabel("S (cts/pixel)")
plt.xlabel("Q (pixels)")

In [ ]:
n=imgs.shape[0]
Ipix=np.zeros((n,len(plist)))
for i in range(n):
    Ipix[i,:]=imgs[i,:,:].ravel()[plist]
print(Ipix.shape,plist.shape)

In [ ]:
ccs,cts=h2t(Ipix,bind,norm="symmetric",Pcorrect=True,rtncounts=True)

In [ ]:
g2s=np.zeros((len(binqlist),ccs.shape[1]))
print(g2s.shape)
for i in range(len(binqlist)):
    g2s[i,:]=avgg2(ccs[i][:,:])

In [ ]:
ccs[50].shape

In [ ]:
plt.figure()
plt.imshow(ccs[10],vmin=.99,vmax=1.2,aspect='auto')
plt.colorbar()